In [1]:
import argparse
import copy

from datasets import load_dataset
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from transformers import GPT2Tokenizer
from tqdm import tqdm
import wandb

from model import MultiLayerRNN, StoryNetwork

In [2]:
def collate_batch(batch, tokenizer, max_length):
    """
    Tokenizes and pads batch of text to longest sequence with ignored padding labels
    
    Args:
        batch: List of examples from dataset
        tokenizer: GPT2 tokenizer instance
        max_length: Maximum sequence length
        
    Returns:
        Dictionary with padded input_ids, labels and attention mask tensors
    """
    texts = [tokenizer.bos_token + example['text'] + tokenizer.eos_token for example in batch]
    
    # First tokenize without padding
    encoded = tokenizer(
        texts,
        truncation=True,
        max_length=max_length + 1,
        return_tensors=None  # Return list of token ids
    )

    input_ids = encoded['input_ids']
    labels = copy.deepcopy(input_ids)
    
    input_ids = [torch.tensor(x, dtype=torch.long) for x in input_ids]
    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)

    labels = [torch.tensor(x, dtype=torch.long) for x in labels]
    labels = pad_sequence(labels, batch_first=True, padding_value=-100)
    
    return {
        'input_ids': input_ids[:, :-1],
        'labels': labels[:, 1:],
        'attention_mask': input_ids.ne(tokenizer.pad_token_id)[:, :-1]
    }

In [3]:
parser = argparse.ArgumentParser()
parser.add_argument('--d_model', type=int, default=512)
parser.add_argument('--batch_size', type=int, default=32)
parser.add_argument('--learning_rate', type=float, default=3e-4)
parser.add_argument('--epochs', type=int, default=10)
parser.add_argument('--eval_every', type=int, default=1)
parser.add_argument('--use_wandb', action='store_true')
parser.add_argument('--max_length', type=int, default=128)
parser.add_argument('--use_normal_gru', dest='use_min_gru', action='store_false')
parser.add_argument('--repeat_sequence', action='store_true', default=False)
parser.add_argument('--two_step_forward', action='store_true', default=False)
parser.add_argument('--model_type', type=str, default='multi_layer')

args = parser.parse_known_args([
    '--d_model', '512',
    '--batch_size', '64',
    '--learning_rate', '3e-4',
    '--epochs', '10',
    '--eval_every', '1',
    '--max_length', '64',
    # '--use_wandb',
])[0]

In [4]:
# Set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
# Load tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load dataset
dataset = load_dataset('roneneldan/TinyStories')
dataset['train'] = dataset['train'].select(range(100000))

# Split into train and validation
val_size = min(1000, int(len(dataset['train']) * 0.1))
train_size = len(dataset['train']) - val_size
train_dataset, val_dataset = random_split(
    dataset['train'], 
    [train_size, val_size]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=args.batch_size,
    shuffle=True,
    num_workers=4,
    collate_fn=lambda b: collate_batch(b, tokenizer, args.max_length)
)

val_loader = DataLoader(
    val_dataset,
    batch_size=args.batch_size,
    shuffle=False,
    num_workers=0,
    collate_fn=lambda b: collate_batch(b, tokenizer, args.max_length)
)

In [5]:
# Experiments:
# 1. Standard GRU on normal sequence
# 2. Standard GRU on repeated sequence
# 3. minGRU on normal sequence
# 4. minGRU on repeated sequence
# 5. minGRU on separate forward pass repeated sequence (forward pass through sequence 2 times with persistant hidden state)
# 6. Same as 5, but modify the hidden state as before
# 7. Same as 5, but integrate hidden state in the same way paper does 


In [6]:
args.repeat_sequence = False
args.two_step_forward = False
args.use_min_gru = True
args.model_type = 'multi_layer'

In [7]:
def sigmoid_linear(x):
    """x < 0 -> sigmoid(x), x >= 0 -> x + 0.5"""
    return torch.where(x >= 0, x + 0.5, x.sigmoid())

def inverse_sigmoid_linear(x):
    """x < 0.5 -> sigmoid(x - 0.5), x >= 0.5 -> x - 0.5"""
    return torch.where(x >= 0.5, x - 0.5, torch.log(x / (1 - x)))


if args.model_type == 'multi_layer':
    model_class = MultiLayerRNN
elif args.model_type == 'story':
    model_class = StoryNetwork


class MemoryNetwork(model_class):
    def __init__(self, *args, **kwargs):
        if 'expansion_factor' not in kwargs:
            kwargs['expansion_factor'] = 2.0
        super().__init__(*args, **kwargs)
        self.memory_dim = int(self.d_model * kwargs['expansion_factor'])
        self.memory_integration_layer = nn.Sequential(
            nn.Linear(self.memory_dim, self.memory_dim),
            nn.ReLU(),
            nn.Linear(self.memory_dim, self.memory_dim),
        )
    
    def modify_future_state(self, future_state: torch.Tensor) -> torch.Tensor:
        return sigmoid_linear(self.memory_integration_layer(future_state))


expansion_factor = 1.0

# Initialize model
model = MemoryNetwork(
    vocab_size=len(tokenizer),
    d_model=args.d_model,
    expansion_factor=expansion_factor,
    use_min_gru=args.use_min_gru,
).to(device)

# Setup optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=args.learning_rate,
)

In [ ]:
# Initialize wandb if requested
if args.use_wandb:
    wandb.init(project='story-mingru-testing-v2')


for epoch in range(args.epochs):
    # Train
    model.train()
    total_loss = 0
    
    batch_losses = []
    batch_accuracies = []
    
    progress = tqdm(train_loader, desc=f'Training Epoch {epoch}')
    for idx, batch in enumerate(progress):
        input_ids = batch['input_ids'].to(device)
        target_ids = batch['labels'].to(device)
        
        
        if args.repeat_sequence:
            input_ids = input_ids.repeat(1, 2)
            target_ids = target_ids.repeat(1, 2)
            target_ids[:, :len(target_ids) // 2] = -100 # Only predict second half of the sequence

        if args.repeat_sequence and args.two_step_forward:
            _, prev_hidden_state = model(input_ids[:, :len(input_ids) // 2])
            logits, _ = model(input_ids[:, len(input_ids) // 2:], prev_hidden_state)
            target_ids = target_ids[:, len(target_ids) // 2:]
        else:
            logits, _ = model(input_ids)


        loss = nn.functional.cross_entropy(
            logits.view(-1, logits.size(-1)), 
            target_ids.reshape(-1),
            ignore_index=-100,
        )
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # Calculate accuracy ignoring padding tokens
        mask = (target_ids != -100)
        correct = (logits.argmax(dim=-1) == target_ids) * mask
        accuracy = correct.sum().float() / mask.sum()
        
        batch_losses.append(loss.item())
        batch_accuracies.append(accuracy.item())

        progress.set_postfix({'loss': loss.item(), 'accuracy': accuracy.item()})
        
        if args.use_wandb and idx % 20 == 0:
            wandb.log({
                'train_loss': torch.tensor(batch_losses).mean().item(),
                'train_accuracy': torch.tensor(batch_accuracies).mean().item(),
                'epoch': epoch,
                'train_step': idx + epoch * len(train_loader),
            })
            batch_losses = []
            batch_accuracies = []

    torch.cuda.empty_cache()
            
    train_loss = total_loss / len(train_loader)
    
    # Evaluate
    if epoch % args.eval_every == 0:
        model.eval()
        total_loss = 0
        total_accuracy = 0
        
        for batch in tqdm(val_loader, desc='Evaluating'):
            input_ids = batch['input_ids'].to(device)
            target_ids = batch['labels'].to(device)
            
            
            if args.repeat_sequence:
                input_ids = input_ids.repeat(1, 2)
                target_ids = target_ids.repeat(1, 2)
                target_ids[:, :len(target_ids) // 2] = -100 # Only predict second half of the sequence

            with torch.no_grad():
                if args.repeat_sequence and args.two_step_forward:
                    _, prev_hidden_state = model(input_ids[:, :len(input_ids) // 2])
                    logits, _ = model(input_ids[:, len(input_ids) // 2:], prev_hidden_state)
                    target_ids = target_ids[:, len(target_ids) // 2:]
                else:
                    logits, _ = model(input_ids)
            
            
            # Calculate loss
            loss = nn.functional.cross_entropy(
                logits.view(-1, logits.size(-1)), 
                target_ids.view(-1),
                ignore_index=-100,
            )
            
            # Calculate accuracy ignoring padding tokens
            mask = (target_ids != -100)
            correct = (logits.argmax(dim=-1) == target_ids) * mask
            accuracy = correct.sum().float() / mask.sum()
            
            total_loss += loss.item()
            total_accuracy += accuracy.item()
            
        val_loss = total_loss / len(val_loader)
        val_accuracy = total_accuracy / len(val_loader)
        
        print(f'\nEpoch {epoch}:')
        print(f'Train Loss: {train_loss:.4f}')
        print(f'Val Loss: {val_loss:.4f}')
        print(f'Val Accuracy: {val_accuracy:.4f}')
        
        if args.use_wandb:
            wandb.log({
                'val_loss': val_loss,
                'val_accuracy': val_accuracy,
                'epoch': epoch
            })